In [ ]:
from sqlalchemy import (
    MetaData,
    Table,
    Column,
    Integer,
    String,
    Boolean,
    Float,
    text,
)
from egomimic.utils.aws.aws_sql import (
    TableRow,
    add_episode,
    update_episode,
    create_default_engine,
    episode_hash_to_table_row,
    delete_episodes,
    episode_table_to_df,
)

In [8]:
engine = create_default_engine()

Tables in schema 'app': ['episodes']


In [9]:
def drop_table(table_name):
    with engine.connect() as connection:
        connection.execute(text(f"DROP TABLE IF EXISTS app.{table_name} CASCADE;"))
        connection.commit()
    print(f"Dropped table '{table_name}' from schema 'app' if it existed.")


# drop_table("episodes2")

In [10]:
def create_table():
    metadata = MetaData(schema="app")

    Table(
        "episodes",
        metadata,
        Column("episode_hash", Integer, primary_key=True),
        Column("operator", String),
        Column("lab", String),
        Column("num_frames", Integer),
        Column("task", String),
        Column("task_description", String),
        Column("scene", String),
        Column(
            "objects", String
        ),  # Store as JSON or comma-separated list of object names
        Column("processed_path", String),
        Column("mp4_path", String),
        Column("embodiment", String),
        Column("robot_name", String),
        Column("is_eval", Boolean),
        Column("eval_score", Float),
        Column("eval_success", Boolean),
    )

    metadata.create_all(engine)
    print("Created table 'episodes' in schema 'app'.")


# create_table()

In [8]:
# Add Episode Test
episode = TableRow(
    episode_hash=123457,
    operator="jdoe",
    lab="lab-x",
    num_frames=1500,
    task="pick_and_place",
    task_description="Dummy row for table inspection",
    scene="kitchen",
    objects="cup,plate,spoon",
    processed_path="/data/processed/abc123",
    mp4_path="/data/videos/abc123.mp4",
    embodiment="ur5",
    robot_name="robby",
    is_eval=True,
    eval_score=0.85,
    eval_success=True,
)

In [9]:
add_episode(engine, episode)

True

In [11]:
df = episode_table_to_df(engine)
df

No rows found in table 'episodes'.


,episode_hash,operator,lab,num_frames,task,task_description,scene,objects,processed_path,mp4_path,embodiment,robot_name,is_eval,eval_score,eval_success


In [7]:
episode.operator = "simar"
update_episode(engine, episode)

True

In [8]:
episode_hash_to_table_row(engine, 123456)

TableRow(episode_hash=123456, operator='simar', lab='lab-x', task='pick_and_place', embodiment='ur5', robot_name='robby', num_frames=1500, task_description='Dummy row for table inspection', scene='kitchen', objects='cup,plate,spoon', processed_path='/data/processed/abc123', mp4_path='/data/videos/abc123.mp4', is_eval=True, eval_score=0.85, eval_success=True)

In [3]:
delete_episodes(engine, [123456, 123457])

True